# Pipeline de traitement des données OpenFoodFacts avec Polars

Ce notebook présente une chaîne de traitement optimisée pour lire et filtrer le fichier CSV complet de OpenFoodFacts (environ 12 Go) sans saturer la mémoire vive (RAM) de l'ordinateur.

**Prérequis** : 
- Avoir téléchargé et décompressé l'archive. Le gros fichier de 12 Go (souvent nommé `en.openfoodfacts.org.products.csv`) doit être déposé dans le même dossier que ce notebook.

## 1. Importation des librairies

In [18]:
import polars as pl
import os

## 2. Définition des paramètres et chargement paresseux (Lazy Loading)

On utilise `pl.scan_csv`, ce qui crée un **plan d'exécution** sans charger concrètement les données en mémoire. C'est l'étape cruciale pour éviter les erreurs `OutOfMemory`.

In [19]:
file_path = "data.csv"
fichier_destination = "produits_openfoodfacts_filtres.csv"

if not os.path.exists(file_path):
    print(f"⚠️ ATTENTION : Le fichier '{file_path}' est introuvable.")
    print("Veuillez le dézipper dans le même répertoire que ce notebook avant de continuer.")
else:
    # Création du plan de lecture en ignorant les erreurs de format mineures
    df_lazy = pl.scan_csv(
        file_path,
        separator='\t',
        ignore_errors=True,
        infer_schema_length=10000,
        low_memory=True,
        quote_char=None 
    )
    print("✅ Plan de lecture paresseux initialisé.")

✅ Plan de lecture paresseux initialisé.


## 3. Définition de la chaîne de traitement (Pipeline)

Nous allons explicitement demander à Polars de :
1. Ne sélectionner **que les colonnes utiles** (la puissance de Polars réside aussi dans sa capacité à utiliser une expression régulière pour cibler les centaines de colonnes nutritionnelles terminant par `_100g`).
2. **Filtrer en amont** en ne conservant que les produits dont le Nutri-score ou l'Éco-score est renseigné, car sans ces valeurs de référence mondiales, un produit est souvent inutile pour nos modèles de Machine Learning.

In [20]:
if os.path.exists(file_path):
    requete = (
        df_lazy
        .select([
            pl.col('code'),
            pl.col('product_name'),
            pl.col('nova_group'),
            pl.col('additives_n'),
            pl.col('^.*_100g$'), # Sélection automatique de tous les nutriments
            pl.col('nutriscore_score'),
            pl.col('nutriscore_grade'),
            pl.col('environmental_score_score'),
            pl.col('environmental_score_grade')
        ])
        .filter(
            # On filtre sur les nouveaux noms de colonnes
            pl.col('nutriscore_score').is_not_null() | pl.col('environmental_score_score').is_not_null()
        )
    )
    print("✅ Requête de filtrage définie.")

✅ Requête de filtrage définie.


## 4. Exécution du traitement en streaming et Sauvegarde finale

La méthode `collect(streaming=True)` indique au moteur en Rust de Polars de lire, traiter et filtrer les 12 Go du fichier **morceau par morceau**. 
À la fin, seul le jeu de données épuré (bien plus petit) sera physiquement chargé dans la RAM de Python et libre pour une utilisation ultérieure par nos modèles.

In [21]:
if os.path.exists(file_path):
    print("⏳ Début du traitement par lots en cascade du fichier... cela peut prendre de 2 à 5 minutes.")
    
    # Exécution réelle de la requête
    df_propre = requete.collect(streaming=True)
    
    print(f"✅ Traitement terminé ! Dimensions du jeu filtré : {df_propre.shape}")
    
    # Sauvegarde de ce nouveau jeu de données propre
    df_propre.write_csv(fichier_destination)
    print(f"💾 Jeu de données sauvegardé localement en tant que : {fichier_destination}")
    
    # Aperçu final
    display(df_propre.head())

⏳ Début du traitement par lots en cascade du fichier... cela peut prendre de 2 à 5 minutes.


C:\Users\Gambey\AppData\Local\Temp\ipykernel_9084\148303692.py:5: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_propre = requete.collect(streaming=True)


✅ Traitement terminé ! Dimensions du jeu filtré : (1563766, 129)
💾 Jeu de données sauvegardé localement en tant que : produits_openfoodfacts_filtres.csv


code,product_name,nova_group,additives_n,energy-kj_100g,energy-kcal_100g,energy_100g,energy-from-fat_100g,fat_100g,saturated-fat_100g,butyric-acid_100g,caproic-acid_100g,caprylic-acid_100g,capric-acid_100g,lauric-acid_100g,myristic-acid_100g,palmitic-acid_100g,stearic-acid_100g,arachidic-acid_100g,behenic-acid_100g,lignoceric-acid_100g,cerotic-acid_100g,montanic-acid_100g,melissic-acid_100g,unsaturated-fat_100g,monounsaturated-fat_100g,omega-9-fat_100g,polyunsaturated-fat_100g,omega-3-fat_100g,omega-6-fat_100g,alpha-linolenic-acid_100g,eicosapentaenoic-acid_100g,docosahexaenoic-acid_100g,linoleic-acid_100g,arachidonic-acid_100g,gamma-linolenic-acid_100g,dihomo-gamma-linolenic-acid_100g,…,chloride_100g,calcium_100g,phosphorus_100g,iron_100g,magnesium_100g,zinc_100g,copper_100g,manganese_100g,fluoride_100g,selenium_100g,chromium_100g,molybdenum_100g,iodine_100g,caffeine_100g,taurine_100g,methylsulfonylmethane_100g,ph_100g,fruits-vegetables-legumes_100g,collagen-meat-protein-ratio_100g,cocoa_100g,chlorophyl_100g,carbon-footprint_100g,glycemic-index_100g,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,nutriscore_score,nutriscore_grade,environmental_score_score,environmental_score_grade
i64,str,i64,i64,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,i64,str,str,str,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,i64,str,str,str,f64,str,str,str,i64,str,str,i64,f64,str,str,str,str,str,str,f64,i64,str,i64,str
7,"""granola Bio le Chocolaté""",null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,4,"""c""",null,"""unknown"""
8,null,4,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6,"""c""",null,"""unknown"""
9,"""xytitol pastilles""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-11,"""a""",null,"""unknown"""
10,"""xxx""",4,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""unknown""",7,"""f"""
13,"""Powdered peanut butter""",null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3,"""c""",null,"""unknown"""
